In [ ]:
# Chunk 0

# === Core Python ===
import os
import glob
import json
import random
import time
from collections import Counter, defaultdict
from typing import Dict, Any, List, Tuple
from pathlib import Path
# === Math & Data ===
import numpy as np
import pandas as pd

# === Visualization ===
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
from PIL import Image

# === Annotation / Dataset Handling ===
import datumaro as dm

import yaml


In [ ]:
BASE_DIR = Path("/scratch/disk4/crab_model_gina/crabs-on-camera/data/annotations") # setting the base directory

CONFIG = {
    "ANNOTATIONS_FILE": BASE_DIR / "test_annotations/annotations/instances_default.json",
    "IMAGES_DIR": BASE_DIR / "test_annotations/images/default",
}


In [ ]:
# Chunk 3 - Annotation stats
# Optional chunk - creates a .csv file with annotation stats for visualization/analysis in R or other programs

def summarize_category_and_species_from_coco(coco_data: dict) -> Counter:
    """Summarize total annotations grouped by category and species."""
    categories = {cat["id"]: cat.get("name", "unknown") for cat in coco_data.get("categories", [])}
    annotations = coco_data.get("annotations", [])

    counts = Counter()
    for ann in annotations:
        cat_id = ann.get("category_id")
        cat_name = categories.get(cat_id, "unknown")

        # Extract species from attributes (case-insensitive)
        species = "unknown"
        if "attributes" in ann and isinstance(ann["attributes"], dict):
            attrs = {k.lower(): str(v).strip().lower() for k, v in ann["attributes"].items()}
            species = attrs.get("species", "unknown")
        if not species or species == "unknown":
            species = cat_name.lower()

        # Clean up for readability
        cat_name = cat_name.strip().title()
        species = species.strip().capitalize()

        counts[(cat_name, species)] += 1

    return counts


def analyze_dataset_to_csv(config: dict, output_csv_path: str = "annotations_summary.csv") -> pd.DataFrame:
    """
    Analyze the COCO-style JSON defined in config and export a summary CSV for R.

    Output columns:
        Category | Species | Count
    """
    ann_path = config["ANNOTATIONS_FILE"]
    print(f"Analyzing dataset")
    print(f"JSON path: {ann_path}")

    with open(ann_path, "r") as f:
        data = json.load(f)

    summary = summarize_category_and_species_from_coco(data)
    total_annotations = len(data.get("annotations", []))
    print(f"Total annotations: {total_annotations:,}")
    print(f"Unique category-species pairs: {len(summary):,}")

    records = [
        {"Category": cat_name, "Species": species, "Count": count}
        for (cat_name, species), count in summary.items()
    ]

    df = pd.DataFrame(records)
    if not df.empty:
        df = df.sort_values(["Category", "Species"]).reset_index(drop=True)

    df.to_csv(output_csv_path, index=False)
    print(f"Exported COCO summary to CSV: {output_csv_path}")
    print(f"Columns: Category | Species | Count")
    print(f"Rows: {len(df):,}")

    return df


summary_df = analyze_dataset_to_csv(CONFIG, output_csv_path="annotations_summary.csv")
display(summary_df.head(10))